In [1]:
pip install evidently psycopg2-binary pandas sqlalchemy

     |████████████████████████████████| 11.7 MB 25.5 MB/s eta 0:00:01
     |████████████████████████████████| 4.2 MB 37.6 MB/s eta 0:00:01
     |████████████████████████████████| 12.8 MB 76.8 MB/s eta 0:00:01
     |████████████████████████████████| 3.2 MB 11.6 MB/s eta 0:00:01     |█████████████▉                  | 1.4 MB 11.6 MB/s eta 0:00:01
     |████████████████████████████████| 57 kB 5.3 MB/s  eta 0:00:01
     |████████████████████████████████| 238 kB 92.5 MB/s eta 0:00:01
     |████████████████████████████████| 19.1 MB 82.2 MB/s eta 0:00:01
     |████████████████████████████████| 79 kB 10.4 MB/s eta 0:00:01
     |████████████████████████████████| 19.5 MB 72.2 MB/s eta 0:00:01B 72.2 MB/s eta 0:00:01
     |████████████████████████████████| 13.5 MB 85.8 MB/s eta 0:00:01
     |████████████████████████████████| 38.6 MB 64.9 MB/s eta 0:00:01��████████▉    | 33.6 MB 64.9 MB/s eta 0:00:01
     |████████████████████████████████| 153 kB 107.7 MB/s eta 0:00:01
     |████████████████████████

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import os

engine = create_engine(
    os.environ["NEON_POSTGRES_URI"]
)

query = "SELECT * FROM model_predictions WHERE date >= now() - interval '1 day'"
current_data = pd.read_sql(query, engine)

KeyError: 'NEON_POSTGRES_URI'

In [ ]:
reference_data = pd.read_sql(
    "SELECT * FROM model_predictions WHERE date BETWEEN '2024-01-01' AND '2024-03-01'",
    engine
)

In [ ]:
from evidently.report import Report
from evidently.metrics import DataDriftPreset

report = Report(metrics=[
    DataDriftPreset()
])

report.run(
    reference_data=reference_data,
    current_data=current_data
)

report.save_html("drift_report.html")

In [ ]:
result = report.as_dict()

if result["metrics"][0]["result"]["dataset_drift"]:
    print("⚠️ Data drift détecté")

In [ ]:
import mlflow

dataset_drift = result["metrics"][0]["result"]["dataset_drift"]
drifted_features = result["metrics"][0]["result"]["number_of_drifted_columns"]
total_features = result["metrics"][0]["result"]["number_of_columns"]

with mlflow.start_run():

    mlflow.log_metric("dataset_drift", int(dataset_drift))
    mlflow.log_metric("drifted_features", drifted_features)
    mlflow.log_metric("total_features", total_features)

In [ ]:
for col, metrics in result["metrics"][0]["result"]["drift_by_columns"].items():
    mlflow.log_metric(f"drift_{col}", int(metrics["drift_detected"]))

In [ ]:
report.save_html("drift_report.html")

In [ ]:
mlflow.log_artifact("drift_report.html")